# Appendix 2. pandas 객체와 핵심 API

## 1. Goal

pandas 코드를 읽으려면 메서드를 외우기 전에 `Series`, `DataFrame`, `Index`가 각각 무엇을 표현하는지 알아야 합니다. 이 notebook은 세 기본 객체에서 출발해 비교와 필터링, 결측값·dtype·문자열 전처리, 범주화, 중복 제거, table 결합과 reshape, 시간 관련 Index, `MultiIndex`, `GroupBy`까지 연결합니다.

예제는 ICU 측정값을 닮은 작은 합성 데이터입니다. API 결과를 확인하는 용도이며 임상적 해석이나 feature 선택에는 사용하지 않습니다.

## 2. Setup

프로젝트 lock의 pandas 버전은 2.3.3입니다.

외부 파일은 읽지 않습니다. cell을 위에서 아래로 실행하면서 객체의 `type`, `index`, `shape`, 결과 label을 함께 확인하세요. 이 notebook에서 `dtype`은 Series나 column이 저장하는 값의 타입, `shape`는 row와 column의 개수, `axis`는 연산 방향이라는 수준으로 먼저 사용합니다. dtype 변환, axis 계산, broadcasting의 자세한 규칙은 다음 NumPy notebook에서 pandas 객체와 비교해 설명합니다.

In [1]:
import pandas as pd

pd.__version__


'2.3.3'

## 3. Steps

각 `###` 절은 하나의 pandas 개념군을, `####` 절은 바로 실행해 확인할 수 있는 세부 학습 단위를 나타냅니다. 기본 객체에서 출발해 특수 Index와 집계 API 순서로 진행합니다.

### 3-1. Series

#### 3-1-1. Series 객체와 구성 요소

`Series`는 한 종류의 값을 담는 1차원 객체입니다. 값 배열뿐 아니라 각 값을 식별하는 `Index`, 전체 값의 의미를 나타내는 `name`, 값의 타입인 `dtype`을 함께 가집니다. 아래에서는 patient별 심박수를 nullable `Float64`로 만듭니다.

In [2]:
patient_index = pd.Index(["P101", "P102", "P103", "P104"], name="patient_id")
heart_rate = pd.Series(
    [78.0, 92.0, pd.NA, 84.0],
    index=patient_index,
    name="heart_rate",
    dtype="Float64",
)
print(
    {
        "type": type(heart_rate).__name__,
        "name": heart_rate.name,
        "dtype": str(heart_rate.dtype),
        "index_name": heart_rate.index.name,
    }
)
heart_rate


{'type': 'Series', 'name': 'heart_rate', 'dtype': 'Float64', 'index_name': 'patient_id'}


patient_id
P101    78.0
P102    92.0
P103    <NA>
P104    84.0
Name: heart_rate, dtype: Float64

#### 3-1-2. Series API: 선택, 결측, 통계와 label 정렬

`loc`는 index label로 값을 선택합니다. `isna`, `fillna`, `dropna`는 결측을 확인하거나 처리하고, `mean`, `min`, `max` 같은 reduction은 Series를 하나의 요약값으로 줄입니다.

pandas 연산의 중요한 특징은 **위치가 아니라 label을 맞춰 계산한다는 것**입니다. 두 Series의 patient 순서나 구성원이 다르면 index의 합집합에 맞춰지고, 한쪽에 없는 label 결과는 결측이 됩니다.

In [3]:
series_summary = {
    "P102_value": heart_rate.loc["P102"],
    "missing_count": int(heart_rate.isna().sum()),
    "observed_mean": float(heart_rate.mean()),
    "filled_values": heart_rate.fillna(heart_rate.median()).tolist(),
}

baseline = pd.Series([78.0, 88.0], index=["P101", "P102"], name="baseline")
current = pd.Series([90.0, 85.0], index=["P102", "P103"], name="current")
aligned_change = current - baseline

print(series_summary)
aligned_change


{'P102_value': np.float64(92.0), 'missing_count': 1, 'observed_mean': 84.66666666666667, 'filled_values': [78.0, 92.0, 84.0, 84.0]}


P101    NaN
P102    2.0
P103    NaN
dtype: float64

### 3-2. DataFrame

#### 3-2-1. DataFrame 객체와 축

`DataFrame`은 행과 열을 가진 2차원 객체입니다. 각 열은 서로 다른 dtype을 가질 수 있는 `Series`이고, 모든 열은 공통 row index를 공유합니다. `index`는 행 label, `columns`도 별도의 `Index` 객체입니다.

In [4]:
measurements = pd.DataFrame(
    {
        "patient_id": ["P101", "P101", "P102", "P102", "P103", "P103", "P104", "P104"],
        "parameter": ["HR", "Lactate", "HR", "Lactate", "HR", "Lactate", "HR", "Lactate"],
        "minute": [15, 30, 10, 60, 20, 45, 25, 90],
        "value": [78.0, 1.8, 92.0, None, 85.0, 3.1, 84.0, 2.2],
    }
)
print(
    {
        "type": type(measurements).__name__,
        "shape": measurements.shape,
        "index_type": type(measurements.index).__name__,
        "column_index_type": type(measurements.columns).__name__,
    }
)
measurements.head()


{'type': 'DataFrame', 'shape': (8, 4), 'index_type': 'RangeIndex', 'column_index_type': 'Index'}


,patient_id,parameter,minute,value
0,P101,HR,15,78.0
1,P101,Lactate,30,1.8
2,P102,HR,10,92.0
3,P102,Lactate,60,NaN
4,P103,HR,20,85.0


#### 3-2-2. DataFrame API: 열, 행, 조건과 파생 열

`df["column"]`은 하나의 Series를, `df[["a", "b"]]`는 DataFrame을 반환합니다. `loc`는 label, `iloc`는 정수 위치를 사용합니다. boolean mask는 조건에 맞는 행을 고르고, `assign`은 원본을 직접 덮어쓰지 않고 파생 열이 추가된 새 DataFrame을 만듭니다.

In [5]:
value_series = measurements["value"]
selected_columns = measurements[["patient_id", "parameter", "value"]]
heart_rate_rows = measurements.loc[measurements["parameter"].eq("HR")].copy()
first_two_rows = measurements.iloc[:2]
prepared_measurements = (
    measurements.assign(
        is_missing=lambda frame: frame["value"].isna(),
        hour_bucket=lambda frame: frame["minute"] // 60,
    )
    .sort_values(["patient_id", "minute"])
)
print(
    {
        "single_column_type": type(value_series).__name__,
        "multiple_columns_type": type(selected_columns).__name__,
        "heart_rate_rows": len(heart_rate_rows),
        "first_two_rows": len(first_two_rows),
    }
)
prepared_measurements


{'single_column_type': 'Series', 'multiple_columns_type': 'DataFrame', 'heart_rate_rows': 4, 'first_two_rows': 2}


,patient_id,parameter,minute,value,is_missing,hour_bucket
0,P101,HR,15,78.0,False,0
1,P101,Lactate,30,1.8,False,0
2,P102,HR,10,92.0,False,0
3,P102,Lactate,60,NaN,True,1
4,P103,HR,20,85.0,False,0
5,P103,Lactate,45,3.1,False,0
6,P104,HR,25,84.0,False,0
7,P104,Lactate,90,2.2,False,1


### 3-3. 비교 연산과 boolean filtering

#### 3-3-1. eq·ne·lt·le·gt·ge로 조건 만들기

비교 메서드는 각 값에 조건을 적용한 boolean Series를 반환합니다. `eq`, `ne`, `lt`, `le`, `gt`, `ge`는 각각 `==`, `!=`, `<`, `<=`, `>`, `>=`와 대응합니다. 메서드 형태는 label alignment와 `fill_value` 같은 pandas 옵션을 함께 사용할 수 있다는 장점이 있습니다.

In [ ]:
raw_patient_rows = pd.DataFrame(
    {
        "patient_id": [" P101 ", "P102", "P103", "P104", "P105", "P102"],
        "unit": ["ICU", " icu ", "WARD", "ICU", None, " icu "],
        "age_text": ["67", "unknown", "52", "41", "89", "unknown"],
        "heart_rate": [78.0, 92.0, 85.0, 110.0, 60.0, 92.0],
        "lactate": [1.8, None, 3.1, 4.2, 2.2, None],
        "observed_at": [
            "2026-01-01 00:10",
            "bad-date",
            "2026-01-01 00:30",
            "2026-01-01 00:40",
            "2026-01-01 00:50",
            "bad-date",
        ],
        "target": [0, 1, 0, 1, 0, 1],
    }
)

comparison_masks = pd.DataFrame(
    {
        "target_eq_1": raw_patient_rows["target"].eq(1),
        "target_ne_0": raw_patient_rows["target"].ne(0),
        "heart_rate_lt_70": raw_patient_rows["heart_rate"].lt(70),
        "heart_rate_le_100": raw_patient_rows["heart_rate"].le(100),
        "heart_rate_gt_90": raw_patient_rows["heart_rate"].gt(90),
        "heart_rate_ge_90": raw_patient_rows["heart_rate"].ge(90),
    }
)
comparison_masks


#### 3-3-2. between·isin과 여러 조건 결합하기

`between(left, right)`는 범위 조건을, `isin(values)`는 허용 목록 조건을 만듭니다. 여러 boolean Series는 `&`, `|`, `~`로 결합하며 각 조건을 괄호로 감쌉니다. Python의 `and`, `or`, `not`은 Series 전체를 하나의 truth value로 판단하려 하므로 사용하지 않습니다.

In [ ]:
expected_heart_rate = raw_patient_rows["heart_rate"].between(
    70,
    100,
    inclusive="both",
)
selected_target = raw_patient_rows["target"].isin([1])
observed_lactate = raw_patient_rows["lactate"].notna()

review_mask = selected_target & (
    ~expected_heart_rate | ~observed_lactate
)
rows_for_review = raw_patient_rows.loc[
    review_mask,
    ["patient_id", "heart_rate", "lactate", "target"],
]
rows_for_review


#### 3-3-3. where·mask·clip으로 값을 조건부 처리하기

`where`는 조건이 True인 값을 유지하고, `mask`는 조건이 True인 값을 바꿉니다. `clip`은 lower와 upper 경계 밖의 값을 경계값으로 제한합니다. 이 API는 원본 값을 조용히 덮어쓰는 대신 새 Series를 반환하므로 전처리 규칙을 비교하기 쉽습니다. 실제 품질 기준은 versioned contract에서 가져와야 합니다.

In [ ]:
heart_rate_values = raw_patient_rows["heart_rate"]
heart_rate_in_range = heart_rate_values.where(
    heart_rate_values.between(40, 200)
)
heart_rate_masked = heart_rate_values.mask(
    heart_rate_values.gt(100)
)
heart_rate_clipped = heart_rate_values.clip(
    lower=40,
    upper=100,
)

pd.DataFrame(
    {
        "original": heart_rate_values,
        "where_40_200": heart_rate_in_range,
        "mask_gt_100": heart_rate_masked,
        "clip_40_100": heart_rate_clipped,
    }
)


### 3-4. 결측값과 dtype 전처리

#### 3-4-1. isna·notna·fillna·dropna

`isna`와 `notna`로 결측 위치를 먼저 확인한 뒤 처리 방법을 정합니다. `fillna`는 결측값을 채우고 `dropna`는 지정한 조건에 맞는 row나 column을 제거합니다. 모든 결측을 같은 값으로 채우지 말고 column의 의미와 이후 계산의 분모를 고려해야 합니다.

In [ ]:
missing_by_column = raw_patient_rows.isna().sum()
lactate_observed_mask = raw_patient_rows["lactate"].notna()
filled_lactate = raw_patient_rows["lactate"].fillna(
    raw_patient_rows["lactate"].median()
)
complete_measurements = raw_patient_rows.dropna(
    subset=["heart_rate", "lactate"]
)

print(
    {
        "missing_by_column": missing_by_column.to_dict(),
        "observed_lactate_rows": int(lactate_observed_mask.sum()),
        "complete_rows": len(complete_measurements),
        "filled_missing_count": int(filled_lactate.isna().sum()),
    }
)


#### 3-4-2. to_numeric·to_datetime·astype·convert_dtypes

외부 데이터의 숫자와 시각은 문자열로 들어오는 경우가 많습니다. `to_numeric`과 `to_datetime`의 `errors="coerce"`는 변환할 수 없는 값을 결측으로 표시해 문제 row를 찾게 해 줍니다. `astype`은 원하는 dtype을 명시하고, `convert_dtypes`는 DataFrame을 pandas nullable dtype 중심으로 정리합니다.

In [ ]:
typed_rows = (
    raw_patient_rows.assign(
        age=lambda frame: pd.to_numeric(
            frame["age_text"],
            errors="coerce",
        ).astype("Int64"),
        observed_at=lambda frame: pd.to_datetime(
            frame["observed_at"],
            errors="coerce",
            format="mixed",
        ),
    )
    .drop(columns="age_text")
    .convert_dtypes()
    .astype({"target": "Int8"})
)

print(
    {
        "dtypes": typed_rows.dtypes.astype(str).to_dict(),
        "invalid_age_rows": int(typed_rows["age"].isna().sum()),
        "invalid_time_rows": int(typed_rows["observed_at"].isna().sum()),
    }
)


### 3-5. 문자열과 범주 전처리

#### 3-5-1. str accessor로 문자열 정리하기

문자열 Series의 `str` accessor는 Python 문자열 메서드를 element-wise로 적용합니다. `strip`, `upper`, `lower`, `replace`, `contains`를 조합해 공백과 표기 차이를 정리하고 원하는 패턴을 찾을 수 있습니다. 결측값은 string dtype 안에서 결측으로 유지합니다.

In [ ]:
cleaned_text_rows = typed_rows.assign(
    patient_id=lambda frame: (
        frame["patient_id"].astype("string").str.strip()
    ),
    unit=lambda frame: (
        frame["unit"]
        .astype("string")
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
        .str.upper()
    ),
)
icu_mask = cleaned_text_rows["unit"].str.contains(
    "ICU",
    na=False,
)

print(
    {
        "patient_ids": cleaned_text_rows["patient_id"].tolist(),
        "units": cleaned_text_rows["unit"].tolist(),
        "icu_rows": int(icu_mask.sum()),
    }
)


#### 3-5-2. map·replace와 category dtype

`map`은 Series의 각 값을 mapping이나 함수로 변환하고, `replace`는 지정한 값을 다른 값으로 바꿉니다. 값의 후보가 제한된 column은 category dtype으로 바꾸면 허용된 categories와 순서를 명시할 수 있습니다. category는 단순 문자열 치환이 아니라 분석에서 사용하는 범주 의미를 표현합니다.

In [ ]:
categorized_rows = cleaned_text_rows.assign(
    target_label=lambda frame: frame["target"].map(
        {0: "negative", 1: "positive"}
    ),
    unit=lambda frame: frame["unit"].replace(
        {"WARD": "GENERAL_WARD"}
    ),
)
categorized_rows["unit"] = categorized_rows["unit"].astype(
    "category"
)

print(
    {
        "target_labels": categorized_rows["target_label"].tolist(),
        "unit_categories": categorized_rows["unit"].cat.categories.tolist(),
    }
)


#### 3-5-3. cut·qcut·get_dummies로 범주 만들기

`cut`은 직접 정한 경계로 연속값을 구간화하고, `qcut`은 관측값의 quantile을 기준으로 비슷한 개수의 구간을 만듭니다. `get_dummies`는 범주를 indicator column으로 펼칩니다. 경계와 reference category는 feature contract의 일부이므로 탐색 결과를 그대로 production default로 사용하지 않습니다.

In [ ]:
age_band = pd.cut(
    categorized_rows["age"],
    bins=[0, 49, 69, 120],
    labels=["under_50", "50_to_69", "70_plus"],
    include_lowest=True,
)
heart_rate_quantile = pd.qcut(
    categorized_rows["heart_rate"],
    q=3,
    labels=False,
    duplicates="drop",
)
unit_indicators = pd.get_dummies(
    categorized_rows["unit"],
    prefix="unit",
    dtype="Int8",
)

print(
    {
        "age_band_counts": age_band.value_counts(dropna=False).to_dict(),
        "heart_rate_quantiles": heart_rate_quantile.tolist(),
        "indicator_columns": unit_indicators.columns.tolist(),
    }
)


### 3-6. Row 정리와 table 재구성

#### 3-6-1. duplicated·drop_duplicates·sort_values·nlargest

`duplicated`는 중복 여부를 boolean Series로 반환하고, `drop_duplicates`는 지정한 key 기준으로 중복 row를 제거합니다. 중복의 의미는 전체 row가 아니라 business key로 정의해야 합니다. `sort_values`와 `nlargest`는 전처리 결과를 검토하거나 상위 관측값을 제한해서 볼 때 유용합니다.

In [ ]:
duplicate_key = ["patient_id", "observed_at"]
duplicate_mask = categorized_rows.duplicated(
    subset=duplicate_key,
    keep=False,
)
deduplicated_rows = (
    categorized_rows.drop_duplicates(
        subset=duplicate_key,
        keep="last",
    )
    .sort_values(["patient_id", "observed_at"], na_position="last")
)
highest_heart_rates = deduplicated_rows.nlargest(
    3,
    columns="heart_rate",
)

print(
    {
        "duplicate_rows": int(duplicate_mask.sum()),
        "deduplicated_rows": len(deduplicated_rows),
        "top_patients": highest_heart_rates["patient_id"].tolist(),
    }
)


#### 3-6-2. merge와 concat으로 table 결합하기

`merge`는 key column을 기준으로 table을 결합하고, `concat`은 같은 schema의 객체를 axis 방향으로 이어 붙입니다. `merge(validate=...)`를 사용하면 예상한 one-to-one 또는 many-to-one 관계가 맞는지 실행 중 확인할 수 있습니다. 결합 전후 row 수와 unmatched key도 함께 점검합니다.

In [ ]:
patient_master = deduplicated_rows[
    ["patient_id", "unit", "age", "heart_rate", "lactate"]
]
risk_scores = pd.DataFrame(
    {
        "patient_id": ["P101", "P102", "P103", "P104", "P105"],
        "risk_score": [0.12, 0.81, 0.34, 0.92, 0.18],
    }
)
merged_patients = patient_master.merge(
    risk_scores,
    on="patient_id",
    how="left",
    validate="one_to_one",
)

new_patient = pd.DataFrame(
    {
        "patient_id": ["P106"],
        "unit": pd.Categorical(["ICU"], categories=patient_master["unit"].cat.categories),
        "age": pd.Series([73], dtype="Int64"),
        "heart_rate": [88.0],
        "lactate": [2.7],
    }
)
all_patients = pd.concat(
    [patient_master, new_patient],
    ignore_index=True,
)

print(
    {
        "merged_rows": len(merged_patients),
        "unmatched_scores": int(merged_patients["risk_score"].isna().sum()),
        "concatenated_rows": len(all_patients),
    }
)


#### 3-6-3. melt와 pivot으로 wide·long 형식 바꾸기

`melt`는 여러 feature column을 variable과 value column으로 내려 long 형식으로 바꿉니다. `pivot`은 unique key 조합을 기준으로 다시 wide 형식을 만듭니다. 한 row가 무엇을 뜻하는지 grain을 먼저 정하고, pivot key가 unique한지 확인한 뒤 변환합니다.

In [ ]:
feature_wide = merged_patients.set_index("patient_id")[
    ["heart_rate", "lactate"]
]
feature_long = (
    feature_wide.reset_index()
    .melt(
        id_vars="patient_id",
        var_name="feature",
        value_name="value",
    )
)
feature_round_trip = feature_long.pivot(
    index="patient_id",
    columns="feature",
    values="value",
).reindex(columns=feature_wide.columns)

print(
    {
        "wide_shape": feature_wide.shape,
        "long_shape": feature_long.shape,
        "round_trip_shape": feature_round_trip.shape,
    }
)
feature_long.head()


### 3-7. Index

#### 3-7-1. Index의 역할과 집합 연산

`Index`는 단순한 row 번호가 아니라 pandas 객체의 축 label입니다. 이름, dtype, uniqueness를 가지며 합집합·교집합·차집합 같은 집합 연산도 제공합니다. Index는 immutable이므로 label 구성을 바꾸는 작업은 새 Index 또는 새 pandas 객체를 반환합니다.

In [6]:
expected_patients = pd.Index(["P101", "P102", "P103", "P104", "P105"], name="patient_id")
observed_patients = pd.Index(measurements["patient_id"].unique(), name="patient_id")
index_operations = {
    "is_unique": observed_patients.is_unique,
    "intersection": expected_patients.intersection(observed_patients).tolist(),
    "missing_patients": expected_patients.difference(observed_patients).tolist(),
    "union": expected_patients.union(observed_patients).tolist(),
}
index_operations


{'is_unique': True,
 'intersection': ['P101', 'P102', 'P103', 'P104'],
 'missing_patients': ['P105'],
 'union': ['P101', 'P102', 'P103', 'P104', 'P105']}

#### 3-7-2. Index API: set_index, reset_index와 reindex

`set_index`는 열을 row label로 올리고, `reset_index`는 다시 일반 열로 내립니다. `reindex`는 기대하는 label 순서와 집합에 맞춰 데이터를 재배열합니다. 원본에 없는 label은 새 행과 결측값으로 드러나므로 coverage 점검에 유용합니다.

In [7]:
patient_summary = (
    prepared_measurements.groupby("patient_id", as_index=False)
    .agg(measurement_count=("value", "size"), missing_count=("is_missing", "sum"))
    .set_index("patient_id")
)
complete_patient_summary = patient_summary.reindex(expected_patients)
round_trip_summary = complete_patient_summary.reset_index()
round_trip_summary


,patient_id,measurement_count,missing_count
0,P101,2.0,0.0
1,P102,2.0,1.0
2,P103,2.0,0.0
3,P104,2.0,0.0
4,P105,NaN,NaN


### 3-8. 시간 관련 Index

#### 3-8-1. DatetimeIndex: 시점 기반 선택과 resample

`DatetimeIndex`는 특정 시점을 label로 표현합니다. `pd.to_datetime`은 문자열을 timestamp로 변환하고, `pd.date_range`는 일정한 간격의 시점을 생성합니다. 시간 index는 정렬한 뒤 사용해야 slicing과 `resample` 결과를 예측하기 쉽습니다. 이 예제는 timezone을 명시해 같은 시각 문자열의 해석 차이도 피합니다.

In [8]:
event_times = pd.date_range(
    "2026-01-01 00:00", periods=6, freq="30min", tz="Asia/Seoul"
)
time_series = pd.Series(
    [78.0, 81.0, 86.0, 84.0, 88.0, 90.0],
    index=event_times,
    name="heart_rate",
).sort_index()
window_start = pd.Timestamp("2026-01-01 00:30", tz="Asia/Seoul")
window_end = pd.Timestamp("2026-01-01 01:30", tz="Asia/Seoul")
selected_window = time_series.loc[window_start:window_end]
hourly_summary = time_series.resample("1h").agg(["count", "mean"])
print({"index_type": type(time_series.index).__name__, "frequency": str(time_series.index.freq)})
hourly_summary


{'index_type': 'DatetimeIndex', 'frequency': '<30 * Minutes>'}


,count,mean
2026-01-01 00:00:00+09:00,2,79.5
2026-01-01 01:00:00+09:00,2,85.0
2026-01-01 02:00:00+09:00,2,89.0


#### 3-8-2. TimedeltaIndex와 PeriodIndex: 경과 시간과 기간

시점과 시간 길이는 다른 개념입니다. `TimedeltaIndex`는 ICU 입실 후 30분처럼 **경과 시간**을 표현하고, `PeriodIndex`는 하루·월·분기처럼 **고정 빈도의 기간**을 표현합니다. 시점이 필요하면 DatetimeIndex, 길이가 필요하면 TimedeltaIndex, 달력 구간이 필요하면 PeriodIndex를 선택합니다.

In [9]:
elapsed_index = pd.to_timedelta(["0min", "30min", "90min", "2h"])
elapsed_series = pd.Series([78.0, 81.0, 86.0, 88.0], index=elapsed_index, name="heart_rate")
daily_periods = pd.period_range("2026-01-01", periods=3, freq="D")
daily_counts = pd.Series([12, 18, 15], index=daily_periods, name="measurement_count")
print(
    {
        "elapsed_index_type": type(elapsed_series.index).__name__,
        "elapsed_minutes": (elapsed_series.index.total_seconds() / 60).tolist(),
        "period_index_type": type(daily_counts.index).__name__,
        "period_frequency": str(daily_counts.index.freq),
    }
)


{'elapsed_index_type': 'TimedeltaIndex', 'elapsed_minutes': [0.0, 30.0, 90.0, 120.0], 'period_index_type': 'PeriodIndex', 'period_frequency': '<Day>'}


### 3-9. MultiIndex

#### 3-9-1. 계층형 축 구성과 level 기반 선택·변형

`MultiIndex`는 하나의 축에 둘 이상의 label level을 둡니다. patient와 parameter 조합처럼 계층적 key를 표현할 때 유용합니다. `loc`는 전체 tuple을, `xs`는 특정 level 값을 선택합니다. `unstack`은 한 index level을 column 축으로 옮겨 긴 형식을 넓은 표로 바꿉니다.

In [10]:
measurement_index = pd.MultiIndex.from_product(
    [["P101", "P102", "P103"], ["HR", "Lactate"]],
    names=["patient_id", "parameter"],
)
latest_values = pd.Series(
    [78.0, 1.8, 92.0, 2.6, 85.0, 3.1],
    index=measurement_index,
    name="latest_value",
)
p101_hr = latest_values.loc[("P101", "HR")]
heart_rate_by_patient = latest_values.xs("HR", level="parameter")
wide_latest_values = latest_values.unstack("parameter")
print({"P101_HR": p101_hr, "level_names": latest_values.index.names})
wide_latest_values


{'P101_HR': np.float64(78.0), 'level_names': FrozenList(['patient_id', 'parameter'])}


parameter,HR,Lactate
patient_id,,
P101,78.0,1.8
P102,92.0,2.6
P103,85.0,3.1


### 3-10. GroupBy

#### 3-10-1. split-apply-combine와 named aggregation

GroupBy는 데이터를 key별로 나누고(split), 각 group에 연산을 적용한 뒤(apply), 결과를 다시 결합합니다(combine). `agg`는 group당 한 행의 요약을 만들며, named aggregation을 사용하면 출력 열 이름과 입력 열·연산을 한곳에서 읽을 수 있습니다.

In [11]:
grouped = prepared_measurements.groupby(["patient_id", "parameter"], sort=True)
group_summary = grouped.agg(
    measurement_count=("value", "size"),
    observed_count=("value", "count"),
    mean_value=("value", "mean"),
    last_minute=("minute", "max"),
)
group_summary


measurement_count  observed_count  mean_value  \
patient_id parameter                                                  
P101       HR                         1               1        78.0   
           Lactate                    1               1         1.8   
P102       HR                         1               1        92.0   
           Lactate                    1               0         NaN   
P103       HR                         1               1        85.0   
           Lactate                    1               1         3.1   
P104       HR                         1               1        84.0   
           Lactate                    1               1         2.2   

                      last_minute  
patient_id parameter               
P101       HR                  15  
           Lactate             30  
P102       HR                  10  
           Lactate             60  
P103       HR                  20  
           Lactate             45  
P104       HR                  25  
           Lactate             90

#### 3-10-2. transform과 filter: 원래 행 유지하기

`agg`가 group을 요약해 행 수를 줄이는 반면, `transform`은 원래 행 수와 index를 유지한 결과를 반환합니다. 따라서 각 측정값에 parameter 평균을 붙이는 작업에 적합합니다. `filter`는 group 전체 조건을 평가해 조건을 만족하는 group의 원본 행만 남깁니다.

In [12]:
parameter_groups = prepared_measurements.groupby("parameter")
measurements_with_group_mean = prepared_measurements.assign(
    parameter_mean=parameter_groups["value"].transform("mean")
)
measurements_with_group_mean["centered_value"] = (
    measurements_with_group_mean["value"]
    - measurements_with_group_mean["parameter_mean"]
)
well_observed_parameters = parameter_groups.filter(
    lambda group: group["value"].count() >= 3
)
print(
    {
        "transform_rows": len(measurements_with_group_mean),
        "filtered_parameters": sorted(well_observed_parameters["parameter"].unique()),
    }
)
measurements_with_group_mean


{'transform_rows': 8, 'filtered_parameters': ['HR', 'Lactate']}


,patient_id,parameter,minute,value,is_missing,hour_bucket,parameter_mean,centered_value
0,P101,HR,15,78.0,False,0,84.750000,-6.750000
1,P101,Lactate,30,1.8,False,0,2.366667,-0.566667
2,P102,HR,10,92.0,False,0,84.750000,7.250000
3,P102,Lactate,60,NaN,True,1,2.366667,NaN
4,P103,HR,20,85.0,False,0,84.750000,0.250000
5,P103,Lactate,45,3.1,False,0,2.366667,0.733333
6,P104,HR,25,84.0,False,0,84.750000,-0.750000
7,P104,Lactate,90,2.2,False,1,2.366667,-0.166667


## 4. Checks

객체 타입, label 정렬, 시간 index, MultiIndex와 GroupBy 결과를 확인합니다. assertion이 실패하면 해당 API가 반환한 객체의 `type`, `shape`, `index`부터 확인하세요.

In [13]:
assert isinstance(heart_rate, pd.Series)
assert isinstance(measurements, pd.DataFrame)
assert isinstance(expected_patients, pd.Index)
assert aligned_change.index.tolist() == ["P101", "P102", "P103"]
assert aligned_change.isna().sum() == 2
assert int(comparison_masks["heart_rate_ge_90"].sum()) == 3
assert int(comparison_masks["heart_rate_lt_70"].sum()) == 1
assert len(rows_for_review) == 3
assert int(heart_rate_masked.isna().sum()) == 1
assert missing_by_column["lactate"] == 2
assert int(typed_rows["age"].isna().sum()) == 2
assert int(typed_rows["observed_at"].isna().sum()) == 2
assert int(icu_mask.sum()) == 4
assert categorized_rows["unit"].cat.categories.tolist() == ["GENERAL_WARD", "ICU"]
assert int(duplicate_mask.sum()) == 2
assert len(deduplicated_rows) == 5
assert len(merged_patients) == 5
assert merged_patients["risk_score"].notna().all()
assert len(all_patients) == 6
assert feature_long.shape == (10, 3)
assert feature_round_trip.shape == feature_wide.shape
assert complete_patient_summary.loc["P105"].isna().all()
assert isinstance(time_series.index, pd.DatetimeIndex)
assert len(selected_window) == 3
assert isinstance(elapsed_series.index, pd.TimedeltaIndex)
assert isinstance(daily_counts.index, pd.PeriodIndex)
assert isinstance(latest_values.index, pd.MultiIndex)
assert latest_values.index.names == ["patient_id", "parameter"]
assert len(group_summary) == 8
assert len(measurements_with_group_mean) == len(prepared_measurements)
print("All appendix checks passed.")


All appendix checks passed.


## 5. Next Steps

- 비교 조건은 `eq`, `lt`, `ge`, `between`, `isin`으로 읽기 쉽게 만들고 `&`, `|`, `~`를 사용할 때는 각 조건을 괄호로 묶습니다.
- 결측값을 처리하기 전에 `isna`로 위치와 개수를 확인하고, `fillna` 또는 `dropna`가 분석 분모에 미치는 영향을 기록합니다.
- 외부 문자열은 `to_numeric`, `to_datetime`, `str` accessor로 정리한 뒤 변환 실패 row를 따로 점검합니다.
- 중복 제거, `merge`, `melt`, `pivot` 전에는 business key와 row grain을 먼저 정의합니다.
- Series와 DataFrame 연산이 예상과 다르면 index label 정렬 여부를 확인합니다.
- 시간 데이터는 시점, 경과 시간, 기간 중 어떤 의미인지 구분한 뒤 적절한 index를 선택합니다.
- MultiIndex는 key 의미가 분명할 때 사용하고, 전달용 표가 필요하면 `reset_index` 또는 `unstack`으로 평탄화합니다.
- GroupBy에서는 원하는 결과가 group당 한 행인지, 원래 행 수를 유지해야 하는지에 따라 `agg`와 `transform`을 구분합니다.
- 다음 notebook에서는 pandas의 dtype, axis, 위치 연산과 label alignment를 NumPy 배열 규칙과 비교합니다.

## 6. References

이 notebook의 설명과 예제는 pandas 2.3.3 공식 문서를 기준으로 작성했습니다.

- [Intro to data structures](https://pandas.pydata.org/pandas-docs/version/2.3.3/user_guide/dsintro.html)
- [Indexing and selecting data](https://pandas.pydata.org/pandas-docs/version/2.3.3/user_guide/indexing.html)
- [Working with missing data](https://pandas.pydata.org/pandas-docs/version/2.3.3/user_guide/missing_data.html)
- [Working with text data](https://pandas.pydata.org/pandas-docs/version/2.3.3/user_guide/text.html)
- [Merge, join, concatenate and compare](https://pandas.pydata.org/pandas-docs/version/2.3.3/user_guide/merging.html)
- [Reshaping and pivot tables](https://pandas.pydata.org/pandas-docs/version/2.3.3/user_guide/reshaping.html)
- [Time series / date functionality](https://pandas.pydata.org/pandas-docs/version/2.3.3/user_guide/timeseries.html)
- [MultiIndex / advanced indexing](https://pandas.pydata.org/pandas-docs/version/2.3.3/user_guide/advanced.html)
- [Group by: split-apply-combine](https://pandas.pydata.org/pandas-docs/version/2.3.3/user_guide/groupby.html)